# Resultados Fase 2 — Subempleo (AOI=03)

**TFG:** Machine Learning explicable para analizar el estado laboral y la calidad del empleo en España.

Este notebook documenta el **cierre académico** de la Fase 2:

1. Baseline y modelo laboral completo (LightGBM)
2. Ablación: solo demografía vs + variables laborales
3. SHAP global y local (waterfalls)
4. Análisis de errores por jornada, sector, ocupación y CCAA

Figuras en `reports/figures/fase2/` y `reports/figures/fase2/cierre/`.


## 1. (Opcional) Regenerar el cierre

Ejecuta la celda siguiente **solo si** quieres volver a calcular ablación, waterfalls y errores. Usa el modelo guardado en `models/fase2/` y puede tardar varios minutos (SHAP). Si ya existen los PNG/JSON en `reports/fase2/cierre/`, salta a la §2.


In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "modelado":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.modeling.fase2_cierre as cierre

cierre = importlib.reload(cierre)
# resumen = cierre.ejecutar_cierre_fase2()
# resumen
print("Descomenta las dos líneas anteriores para regenerar el cierre.")


## 2. Resumen numérico


In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "modelado":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

cierre_json = ROOT / "reports/fase2/cierre/cierre_prioridad1.json"
comp_json = ROOT / "reports/fase2/comparacion_modelos.json"
maestras_json = ROOT / "reports/memoria/metricas_maestras.json"

with open(comp_json, encoding="utf-8") as f:
    full = json.load(f)
with open(maestras_json, encoding="utf-8") as f:
    maestras = json.load(f)

print("=== MODELO COMPLETO (LightGBM) ===")
print(f"Ganador: {full['ganador']}")
print(f"PR-AUC: {full['metricas_test_umbral_05']['pr_auc']:.4f}")
print(f"ROC-AUC: {full['metricas_test_umbral_05']['roc_auc']:.4f}")
print(f"Umbral F1 (train): {full['umbral_optimo_f1_train']:.3f}")
print(f"F1 @ umbral óptimo: {full['metricas_test_umbral_optimo']['f1']:.4f}")
print(f"Tasa subempleo global: {full['tasa_subempleo_global']*100:.2f} %")
print()

if cierre_json.exists():
    with open(cierre_json, encoding="utf-8") as f:
        datos = json.load(f)
    ab = datos["ablacion"]
    print("=== ABLACIÓN (PR-AUC test) ===")
    print(f"Baseline:        {ab['baseline_prevalencia']['pr_auc']:.4f}")
    print(f"Solo demografía: {ab['solo_demografia']['pr_auc']:.4f}")
    print(f"Completo:        {ab['modelo_completo']['pr_auc']:.4f}")
    print(f"Salto:           +{ab['modelo_completo']['pr_auc'] - ab['solo_demografia']['pr_auc']:.4f}")
else:
    ab = maestras["fase2"]["ablacion"]
    print("=== ABLACIÓN (desde metricas_maestras.json) ===")
    print(f"Baseline:        {ab['baseline_pr_auc']:.4f}")
    print(f"Solo demografía: {ab['solo_demografia_pr_auc']:.4f}")
    print(f"Completo:        {ab['completo_pr_auc']:.4f}")
    print(f"Salto:           +{ab['salto_pr_auc']:.4f}")

resumen_md = ROOT / "reports/fase2/cierre/resumen_cierre.md"
if resumen_md.exists():
    display(Markdown(resumen_md.read_text(encoding="utf-8")))


## 3. Galería de figuras


In [ ]:
from IPython.display import Image, Markdown, display


def _mostrar_fig(path):
    if not path.exists():
        display(Markdown(f"*No encontrada:* `{path.relative_to(ROOT)}`"))
        return
    display(Markdown(f"#### `{path.name}`"))
    display(Image(data=path.read_bytes(), width=720))


# reports/figures/fase2

_mostrar_fig(ROOT / "reports/figures/fase2" / "curva_pr.png")

_mostrar_fig(ROOT / "reports/figures/fase2" / "matriz_confusion_umbral_05.png")

_mostrar_fig(ROOT / "reports/figures/fase2" / "matriz_confusion_umbral_optimo.png")

_mostrar_fig(ROOT / "reports/figures/fase2" / "shap_summary.png")

_mostrar_fig(ROOT / "reports/figures/fase2" / "shap_importancia_barras.png")

# reports/figures/fase2/cierre

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "ablacion_demografia_vs_laboral.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "errores_por_jornada.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "errores_por_sector.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "errores_por_ocupacion.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "errores_por_ccaa.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "shap_waterfall_tp_subempleo.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "shap_waterfall_tn_no_subempleo.png")

_mostrar_fig(ROOT / "reports/figures/fase2/cierre" / "shap_waterfall_fn_subempleo.png")


## 4. Decisiones que ilustra este cierre

| Decisión | Evidencia |
|---|---|
| Variables laborales aportan | Ablación: PR-AUC demografía << completo (+≈0,32) |
| Anti-leakage (sin MASHOR/DISMAS/HORDES/BUSOTR) | Target AOI=03 oficial; horas como riesgo, no definición |
| Umbral calibrado F1 (0,825) | Matriz @0,825 vs @0,50 |
| XAI accionable | Waterfalls TP / TN / FN |
| Lectura de calidad del empleo | Errores por jornada, sector, ocupación, CCAA |

Con esto la **Fase 2** queda documentada para memoria y defensa.
